# Keras MNIST CNN Tutorial

## What makes Keras different from PyTorch?

Both Keras (backed by TensorFlow) and PyTorch are Python libraries for building and training
neural networks. They do the same job — but they make very different trade-offs between
convenience and control.

**PyTorch** exposes every step of the training process explicitly. The training loop requires
four lines per batch: `optimizer.zero_grad()` → forward pass → `loss.backward()` →
`optimizer.step()`. Shapes are never inferred — you compute and hard-code every layer input
size yourself. Nothing uses the GPU unless you call `.to(device)`. Images are stored
channels-first: `(channels, height, width)`.

**Keras** was designed to let you build a working model in as few lines as possible. It
handles the training loop invisibly inside `model.fit()`, infers shapes automatically from
`keras.Input`, uses the GPU automatically if one is present, and stores images
channels-last: `(height, width, channels)`.

## Why this tutorial exists

If you're coming from PyTorch, four things will immediately stand out:

1. **There is no explicit training loop** — `model.fit(x_train, y_train, epochs=3)` replaces
   the entire `zero_grad → forward → backward → step` cycle. Keras runs it invisibly.

2. **Shapes are inferred for you** — define a `Dense` layer with just the output size; Keras
   figures out the input size automatically from whatever came before. No `64 * 5 * 5`
   arithmetic to compute by hand.

3. **The GPU is used automatically** — TensorFlow detects a GPU and uses it with no
   `.to(device)` calls.

4. **Images are channels-last, not channels-first** — `(height, width, channels)` vs.
   PyTorch's `(channels, height, width)`. Mixing them up can silently produce wrong results.

This notebook builds **the same MNIST CNN you know from PyTorch — purely in Keras**, so you
can see exactly what Keras hides and why that trade-off is sometimes worth it.

## What you'll build

A small **convolutional neural network** (CNN) on MNIST:

```
Input (28×28×1) → Conv 32 → MaxPool → Conv 64 → MaxPool → Flatten → Dense 128 → Dense 10
```

## What you'll be able to do when done

- Write `model.compile()` + `model.fit()` understanding what each hides vs. the PyTorch loop
- Explain why Keras images are `(28, 28, 1)` while PyTorch uses `(1, 28, 28)`
- Save and reload weights using Keras idioms vs. PyTorch's `state_dict()`
- Know when Keras convenience is worth it vs. when to reach for a custom training loop

## Keras and the Training Cycle

Training a neural network is a five-step cycle that repeats for every batch:

1. **Load a batch** — grab the next chunk of training examples
2. **Forward pass** — run the batch through the model to get predictions
3. **Compute the loss** — measure how wrong the predictions were
4. **Backward pass** — compute gradients via backpropagation
5. **Update the weights** — apply the optimizer update

**Keras runs all five steps inside `model.fit()`** — you hand it your dataset and epochs,
and the loop runs invisibly.

**PyTorch exposes all five steps as separate lines of code:**

```python
# The four PyTorch lines — Keras hides all of these inside model.fit():
optimizer.zero_grad()          # clear accumulated gradients
outputs = model(images)        # forward pass
loss = criterion(outputs, labels)  # compute loss
loss.backward()                # backward pass
optimizer.step()               # update weights
```

The same five things happen in both frameworks. The difference is that Keras hides
them — which is convenient until something goes wrong and you need to see inside.

## Table of Contents

1. [Roadmap](#roadmap)
2. [Section 1: Imports & Seeding](#section-1-imports--seeding)
3. [Section 2: Loading MNIST](#section-2-loading-mnist)
4. [Section 3: Defining the Model](#section-3-defining-the-model)
5. [Section 4: Compile](#section-4-compile)
6. [Section 5: Training](#section-5-training)
7. [Section 6: Evaluation](#section-6-evaluation)
8. [Section 7: Inference / Prediction](#section-7-inference--prediction)
9. [Saving and Loading Weights](#saving-and-loading-weights)
10. [Cheat Sheet: PyTorch → Keras](#cheat-sheet-pytorch--keras)
11. [What This Notebook Covered (and What It Didn't)](#what-this-notebook-covered-and-what-it-didnt)
12. [Roadmap — completed](#roadmap--completed)

## Roadmap

| Step | Concept | Key Idea |
|---|---|---|
| 1 | Imports & seeding | `tf.random.set_seed(42)` + `np.random.seed(42)` — two seeds because TF and NumPy have separate generators |
| 2 | Loading MNIST | `keras.datasets.mnist` returns NumPy arrays directly — no `Dataset`/`DataLoader` needed |
| 3 | Defining the model | `keras.Sequential` infers every shape automatically — no `64*5*5` arithmetic by hand |
| 4 | Compile | `model.compile()` bundles optimizer, loss, and metrics — replaces PyTorch's separate `criterion` + `optimizer` objects |
| 5 | Training | `model.fit()` hides the entire loop — replaces PyTorch's explicit `zero_grad → forward → backward → step` |
| 6 | Evaluation | `model.evaluate()` — one line; no `model.eval()` + `torch.no_grad()` needed |
| 7 | Inference | `model.predict()` returns probabilities — no `argmax(logits)` needed |
| — | Save/load + cheat sheet | `save_weights` / `load_weights` vs. PyTorch's `state_dict()` round-trip |

## Section 1: Imports & Seeding

### Two seed calls, one for each library

Keras uses **TensorFlow** for all neural network math, and **NumPy** separately to prepare
data. These are two independent libraries, each with their own random number generator:

```python
tf.random.set_seed(42)   # seeds TensorFlow's generator — controls weight init, dropout, etc.
np.random.seed(42)        # seeds NumPy's generator — controls data shuffles and preprocessing
```

**PyTorch** uses a single generator across the whole ecosystem, so one call covers everything:
`torch.manual_seed(42)`. Keras/TF requires two.

### GPU — automatic in Keras, explicit in PyTorch

**Keras/TensorFlow detects a GPU automatically and uses it without any action from you.**
There are no `.to(device)` calls. TensorFlow decides which device to use internally, and
you can check whether a GPU was found with `tf.config.list_physical_devices('GPU')`.

In PyTorch, every tensor starts on the CPU and you move models and data explicitly:
`model.to(device)` and `images.to(device)` — forgetting either raises a `RuntimeError`.
This forced explicitness is intentional; Keras's automatic placement can hide performance
problems.

In [ ]:
#  Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy",      "numpy"),
    ("matplotlib", "matplotlib"),
    ("tensorflow", "tensorflow"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices("GPU")
device_str = "GPU" if gpus else "CPU"
print("TensorFlow version:", tf.__version__)
print(f"TensorFlow will use: {device_str}")
print("No .to(device) calls needed — Keras uses the GPU automatically if present.")

## Section 2: Loading MNIST

`keras.datasets.mnist` hands you plain NumPy arrays — `(x_train, y_train), (x_test, y_test)`
— already split into train/test. No `Dataset`/`DataLoader` concept; you pass the arrays
straight into `model.fit()` and Keras batches them internally.

**PyTorch** wraps data in a `Dataset` + `DataLoader` two-layer abstraction. The `DataLoader`
handles batching, shuffling, and iteration. Keras does all of this implicitly inside
`model.fit(batch_size=128, shuffle=True)`.

### The channel-axis difference

Keras stores image batches as `(batch, height, width, channels)` — **channels last (NHWC)**.
PyTorch convolution layers expect `(batch, channels, height, width)` — **channels first (NCHW)**.

The pixel values are identical; only the axis order differs. `torchvision.transforms.ToTensor()`
handles the NHWC→NCHW conversion automatically. In Keras you just add the channel dimension last.

### Predict first

Before running the loading cell below: both Keras and PyTorch load the exact same 60,000
training MNIST images. After loading, each framework prints the shape of one image tensor.

- What shape do you think **Keras** will print for one image? (channels last)
- What shape would **PyTorch** print for the same image? (channels first)
- Are the dimensions in the same order?

Run the cell and compare against your guess.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

# Add channel dimension LAST (NHWC) → shape (N, 28, 28, 1)
x_train = x_train[..., np.newaxis]
x_test  = x_test[..., np.newaxis]

print("x_train:", x_train.shape, "| y_train:", y_train.shape)
print("x_test: ", x_test.shape,  " | y_test: ", y_test.shape)
print("\nKeras: (N, H, W, C) — channels LAST.")
print("PyTorch equivalent would be: (N, C, H, W) — channels FIRST → (60000, 1, 28, 28).")

In [ ]:
# Visualize a few training examples
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i].squeeze(), cmap="gray")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.suptitle("Sample MNIST digits (60,000 total training images)")
plt.tight_layout()
plt.show()

### The gotcha: NHWC vs. NCHW

The two shapes — Keras `(28, 28, 1)` vs. PyTorch `(1, 28, 28)` — are not a typo; they
represent the same image with axes in different orders. This is the #1 silent shape bug
when porting a preprocessing pipeline between the two frameworks: the data loads, the
model trains, but the convolution runs on the wrong axis and accuracy drops for no
visible reason.

Keras `Conv2D` expects `(H, W, C)` per image — channels last. PyTorch `nn.Conv2d` expects
`(C, H, W)` — channels first. Keep this difference in mind any time you move NumPy arrays
between frameworks.

## Section 3: Defining the Model

**Keras:** `keras.Sequential` lists layers in order. Declare the input shape once via
`keras.Input`; every layer after that infers its input size automatically — including
the flattened size going into the first `Dense` layer. You never compute `64 * 5 * 5`.

**PyTorch:** you subclass `nn.Module`, declare every layer in `__init__` with an explicit
`in_features`, and write `forward()` yourself. The flatten arithmetic (`64 * 5 * 5 = 1600`)
must be computed and hard-coded; getting it wrong produces a `RuntimeError` on the first
`forward()` call, not at definition time.

### Predict first

The Keras model below never mentions a flattened size. PyTorch would require you to compute
it by hand: `28 → conv3 → 26 → pool2 → 13 → conv3 → 11 → pool2 → 5`, then `64 × 5 × 5 = 1600`.

Keras's `Flatten()` + the first `Dense` layer figure it out from `keras.Input(shape=(28,28,1))`
automatically. No arithmetic needed.

In [ ]:
keras_model = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(64, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
print("Keras Sequential defined.")
print(f"Total parameters: {keras_model.count_params():,}")
print("Flatten() + Dense() inferred the flattened size automatically — no 64*5*5 needed.")

In [ ]:
# Keras model.summary() shows layer names, output shapes, and parameter counts for free.
# PyTorch equivalent: print(model) shows the module tree (no shapes), plus a manual
# sum(p.numel() for p in model.parameters()) for the count.
keras_model.summary()

#### What just happened

The model was defined without any flatten-size arithmetic. `keras.Input(shape=(28,28,1))`
propagated the shape through every layer automatically, so `Flatten()` already knows its
output size is `1600` and `Dense(128)` already knows its input size is `1600` — none of
this was ever written down or computed.

In PyTorch, `nn.Linear(64 * 5 * 5, 128)` forces you to write that `1600` explicitly. The
dummy-forward-pass verification pattern in the PyTorch primer exists precisely to catch a
wrong number there. Keras eliminates the problem entirely.

## Section 4: Compile

**Keras:** `model.compile()` bundles the optimizer, loss function, and evaluation metrics
into the model object. `model.fit()` reads all three from there automatically.

**PyTorch:** there is no `compile()`. The loss function (`criterion`) and optimizer are
plain objects you create and hold onto yourself — you must use them correctly inside
the training loop:

```python
# PyTorch — two separate objects, used manually in the loop:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

Note: Keras's `"sparse_categorical_crossentropy"` loss matches PyTorch's `nn.CrossEntropyLoss()`
when labels are plain integers (not one-hot). The `from_logits=False` default means Keras
expects probabilities from the final `softmax` layer — whereas PyTorch's `CrossEntropyLoss`
expects raw logits and applies `log_softmax` internally.

In [ ]:
keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",  # labels are plain ints, not one-hot
    metrics=["accuracy"],
)
print("Model compiled — optimizer, loss, and metrics bundled into the model object.")
print("model.fit() will read all three from here automatically.")

## Section 5: Training

This is the single biggest difference between the two frameworks.

**Keras:** `model.fit()` hides the entire training loop — batching, forward pass, backward
pass, optimizer step, and metric tracking are all handled internally.

**PyTorch:** you write the loop over epochs and batches yourself. Four explicit lines do
the actual learning, and Keras users almost always forget the first one at least once:

```python
# The four PyTorch lines that model.fit() replaces:
optimizer.zero_grad()          # 1. clear accumulated gradients (forget this → silent corruption)
outputs = model(images)        # 2. forward pass
loss = criterion(outputs, labels)  # 3. compute loss
loss.backward()                # 4. backward pass
optimizer.step()               # 5. update weights
```

The `validation_split=0.1` argument reserves 10% of training data for validation metrics
at the end of each epoch — equivalent to holding out a val split and manually evaluating
it after every training epoch in PyTorch.

In [ ]:
history = keras_model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=3,
    validation_split=0.1,
)

#### What just happened

`model.fit()` ran 3 epochs over 54,000 training images (10% reserved for validation),
printing loss and accuracy after each epoch. Internally it:

1. Shuffled the data and split it into 128-image batches
2. For each batch: ran the forward pass, computed cross-entropy loss, ran backpropagation
   via TensorFlow's automatic differentiation, applied the Adam update
3. At epoch end: evaluated the 6,000 validation images

The `history` object returned contains loss and accuracy curves — equivalent to appending
`loss.item()` to a list in the PyTorch manual loop.

In [ ]:
# Plot the training history (Keras returns a History object automatically)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curve")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy curve")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Keras returns a History object from model.fit() — equivalent to manually tracking")
print("loss_history.append(loss.item()) per epoch in the PyTorch loop.")

### `model.train()` / `model.eval()` — does Keras need them?

In PyTorch, you must explicitly call `model.train()` before the training loop and
`model.eval()` + `torch.no_grad()` before evaluation — otherwise `Dropout` and `BatchNorm`
layers use the wrong mode, silently changing their behavior.

**Keras handles this automatically:**

- `model.fit()` switches all layers to training mode internally on each batch
- `model.evaluate()` and `model.predict()` switch to inference mode automatically
- You never call a Keras equivalent of `model.train()` or `model.eval()`

If you ever write a **custom Keras training loop** using `tf.GradientTape`, you do need
to pass `training=True` or `training=False` to `model(x, training=...)` explicitly —
that's the Keras equivalent of PyTorch's `model.train()` / `model.eval()` call.

## Section 6: Evaluation

**Keras:** `model.evaluate()` runs the test set through the compiled loss and metrics in
one call. No mode-switching, no `torch.no_grad()` context manager — Keras handles all of
that internally.

**PyTorch equivalent** — what Keras hides:

```python
model.eval()          # switch off Dropout / BatchNorm training behavior
with torch.no_grad(): # disable gradient tracking — saves memory, speeds up inference
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # accumulate loss and accuracy manually...
```

In [ ]:
test_loss, test_acc = keras_model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print("\nNo model.eval() or torch.no_grad() calls needed — Keras handles mode-switching")
print("and gradient-tape disabling internally inside evaluate().")

## Section 7: Inference / Prediction

**Keras:** `model.predict()` runs the forward pass and returns **probabilities** — because
the final layer has `activation="softmax"`. The highest probability is the prediction.

**PyTorch:** `SimpleCNN.forward()` returns raw logits with no softmax. You call
`logits.argmax(dim=1)` to get the predicted class. `argmax` on logits gives the same class
as `argmax` on softmax-probabilities (softmax is strictly increasing), so an explicit
`softmax` layer is never needed — unless you need the actual probability values for display.

**Why the difference?** PyTorch's `nn.CrossEntropyLoss` applies log-softmax internally,
so the model's `forward()` should output raw logits. Keras's
`"sparse_categorical_crossentropy"` expects probabilities (after softmax), so the model's
last layer includes `activation="softmax"`.

In [ ]:
sample_x = x_test[:8]
sample_y = y_test[:8]

# model.predict() returns probabilities (after softmax) — shape (8, 10)
probs = keras_model.predict(sample_x, verbose=0)
pred_labels = probs.argmax(axis=1)

print("True:      ", sample_y.tolist())
print("Predicted: ", pred_labels.tolist())

In [ ]:
# Visualize predictions on the 8 sample digits
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(sample_x[i].squeeze(), cmap="gray")
    color = "green" if pred_labels[i] == sample_y[i] else "red"
    ax.set_title(f"{pred_labels[i]}", color=color)
    ax.axis("off")
plt.suptitle("Keras predictions (green=correct, red=wrong)")
plt.tight_layout()
plt.show()

print(f"\nTest accuracy: {test_acc:.4f} over the full 10,000-image test set.")

## Saving and Loading Weights

**Keras:** `model.save_weights(path)` saves all trainable tensors to a file.
`model.load_weights(path)` restores them into a matching architecture. There is also
`model.save(path)` which saves the entire model (architecture + weights + optimizer state)
in a single file — Keras can reconstruct the model from that file alone.

**PyTorch:** the idiomatic pattern is weights-only. `model.state_dict()` returns an ordered
dict of every learnable tensor by name, saved with `torch.save`. Loading requires:
1. Re-instantiate the same `nn.Module` class (architecture is not saved)
2. Call `model.load_state_dict(torch.load(path, weights_only=True))`

The cell below proves this with a real round-trip: save `keras_model`'s weights, load them
into a **freshly constructed, randomly-initialized** model, and confirm its predictions on
the same sample images now match the trained model's exactly.

In [ ]:
import tempfile, os

# Save weights
with tempfile.NamedTemporaryFile(suffix=".weights.h5", delete=False) as f:
    weights_path = f.name

keras_model.save_weights(weights_path)
print(f"Saved weights to {weights_path}")

# Fresh model with the same architecture — randomly initialized, NOT a copy
fresh_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
fresh_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Predictions before loading — random weights → wrong answers
fresh_preds_before = fresh_model.predict(sample_x, verbose=0).argmax(axis=1)

# Load the trained weights
fresh_model.load_weights(weights_path)
fresh_preds_after = fresh_model.predict(sample_x, verbose=0).argmax(axis=1)

os.unlink(weights_path)

print(f"\nFresh (random) model:   {fresh_preds_before.tolist()}")
print(f"After load_weights():   {fresh_preds_after.tolist()}")
print(f"Original keras_model:   {pred_labels.tolist()}")
assert (fresh_preds_after == pred_labels).all(), "Loaded weights must reproduce original predictions."
print("\n-> Confirmed: load_weights() reproduced the trained model's exact predictions.")
print("   In PyTorch you'd use model.load_state_dict(torch.load(path, weights_only=True))")
print("   on a freshly-instantiated SimpleCNN — same concept, different API.")

## Cheat Sheet: PyTorch → Keras

| PyTorch concept | Keras equivalent |
| --- | --- |
| `class Model(nn.Module)` with `__init__` + `forward()` | `keras.Sequential([...])` or `keras.Model` subclass with `call()` |
| Compute every layer's input size by hand | Automatic shape inference from `keras.Input` |
| `nn.Linear(in_features, out_features)` | `layers.Dense(units)` — no `in_features` needed |
| `nn.Conv2d(in_channels, out_channels, kernel_size)` | `layers.Conv2D(filters, kernel_size)` |
| `nn.MaxPool2d(kernel_size)` | `layers.MaxPooling2D(pool_size)` |
| `nn.Flatten()` / `x.view(x.size(0), -1)` | `layers.Flatten()` |
| `Dataset` + `DataLoader`, shape `(N, C, H, W)` channels-first | NumPy arrays, shape `(N, H, W, C)` channels-last |
| `criterion = nn.CrossEntropyLoss()` + `optimizer = Adam(...)` | `model.compile(optimizer=..., loss=..., metrics=[...])` |
| Manual loop: `zero_grad → forward → backward → step` | `model.fit(x_train, y_train, epochs=...)` |
| Manual loop with `model.eval()` + `torch.no_grad()` | `model.evaluate(x_test, y_test)` |
| `model(x)` inside `torch.no_grad()` after `model.eval()` | `model.predict(x)` — returns probabilities, not logits |
| `print(model)` + `sum(p.numel() for p in model.parameters())` | `model.summary()` |
| Explicit `.to(device)` on model and every batch | GPU used automatically — no `.to(device)` calls |
| Final `nn.Linear` returns raw logits | Final `Dense(10, activation="softmax")` returns probabilities |
| `torch.save(model.state_dict(), path)` | `model.save_weights(path)` or `model.save(path)` |
| `model.load_state_dict(torch.load(path))` on re-instantiated class | `model.load_weights(path)` |
| `model.train()` / `model.eval()` — must call explicitly | Handled automatically by `fit()` / `evaluate()` / `predict()` |

## Gotchas worth remembering

1. **Channels-last, not channels-first.** `(H, W, C)` per image in Keras vs. `(C, H, W)` in
   PyTorch. This is the #1 shape-mismatch bug when porting data pipelines between frameworks.
2. **Loss + softmax interaction.** Keras's `"sparse_categorical_crossentropy"` expects
   probabilities (softmax already applied), so your last layer needs `activation="softmax"`.
   PyTorch's `nn.CrossEntropyLoss` applies log-softmax internally, so your last layer
   returns raw logits with no activation.
3. **Custom training loops need explicit `training=` flags.** If you use `tf.GradientTape`
   instead of `model.fit()`, pass `training=True` inside the tape and `training=False` for
   evaluation — this is how Keras controls Dropout and BatchNorm mode in custom loops.
4. **`model.fit()` shuffles by default.** Unlike PyTorch's `DataLoader(shuffle=False)`,
   Keras shuffles the training data every epoch unless you set `shuffle=False`.
5. **Weight saving needs a compiled model.** Call `model.compile(...)` before
   `model.load_weights(...)` so the model knows its optimizer state when resuming training.

## Next steps

Head to [`01-rnns/`](../01-rnns/) next — the same Keras patterns now applied to sequence
data: recurrent networks, LSTMs, and `tf.GradientTape` for custom training.
The Keras version of the intro notebook is [`PT-Part1-Intro.ipynb`](../01-rnns/PT-Part1-Intro.ipynb).

## What This Notebook Covered (and What It Didn't)

The **Roadmap — completed** table below has the full side-by-side list of what was
actually built and measured.

A couple of extra ideas got a quick illustration along the way, and related topics
were named but deliberately left out:

- **Illustrated in a short note, not a full section:** `training=` mode in custom Keras
  loops, `model.save()` vs. `model.save_weights()`, the `History` object from `model.fit()`.
- **Named, but out of scope:** `tf.GradientTape` custom training loops, writing a custom
  `tf.data.Dataset`, data augmentation with `keras.layers.RandomFlip` / `RandomRotation`,
  learning-rate schedulers and Keras callbacks, `tf.saved_model` export for serving,
  `tf.keras.mixed_precision` for GPU training speed.

## Roadmap — completed

| Step | Concept | Confirmed by |
|---|---|---|
| 1 | Imports & seeding | TF version + device printed from one seeding call each |
| 2 | Loading MNIST | Measured shapes showed channels-last `(28, 28, 1)`, not channels-first |
| 3 | Defining the model | `model.summary()` showed automatically-inferred flattened size — no `64*5*5` arithmetic |
| 4 | Compile | `Adam(lr=1e-3)` + cross-entropy loss bundled in one `compile()` call |
| 5 | Training | `model.fit()` ran 3 epochs; `History` object returned loss + accuracy curves |
| 6 | Evaluation | One-line `evaluate()` returned test loss and accuracy |
| 7 | Inference | `model.predict()` returned probabilities; `argmax` selected the predicted digit |
| — | Save/load | `save_weights` + `load_weights` on a fresh model reproduced exact predictions |

## Key insights to keep

- `model.fit()` and the four-line PyTorch loop (`zero_grad → forward → backward → step`)
  do the same work — `model.fit()` just makes every step invisible.
- Keras never makes you compute a layer's input size: `keras.Input` propagates shapes
  through the entire architecture automatically.
- Channels-last vs. channels-first is the single most common silent bug when moving
  a data pipeline between Keras and PyTorch — always check `.shape` after loading.
- The softmax/logits split: Keras's last layer has `activation="softmax"` → probabilities;
  PyTorch's last layer returns raw logits → `CrossEntropyLoss` applies log-softmax internally.

---

## When to Use Which Framework — and Which Keras Pattern

### Keras vs. PyTorch: pick based on task

| Situation | Reach for | Reason |
|---|---|---|
| Quick prototype, standard architecture, no custom grad | Keras / `model.fit()` | Less boilerplate; validation, callbacks, logging built in |
| Custom training loop, research, gradient surgery | PyTorch explicit loop | Full control; every `zero_grad`, `backward`, `step` is visible |
| Reading a research paper that ships PyTorch code | PyTorch | ~90% of published deep-learning code is PyTorch-first |
| Production deployment on TF-serving / TFLite / TF.js | Keras | Better TF ecosystem integration |
| Fine-tuning a HuggingFace model | PyTorch (PEFT, Trainer) | HuggingFace ecosystem is PyTorch-native |

### Keras architecture pattern: pick based on complexity

| Model complexity | Pattern | When to move up |
|---|---|---|
| ≤5 standard layers, no branches | `keras.Sequential` | When you need a custom `call()` method |
| Custom logic, skip connections, conditional paths | `keras.Model` subclass | Always safe; use for everything complex |
| Full manual control over the gradient step | `tf.GradientTape` custom loop | When `model.fit()` doesn't expose what you need |